In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
import pickle

In [30]:
df = pd.read_csv('../data/solar_fault_monitor_preprocessed.csv')

In [31]:
df.head()

,temperature_c,ldr_left,ldr_right,avg_light,light_diff,voltage_v,current_a,power_w,fault_type,split
0,69.74,841.98,891.24,866.61,49.26,15.37,3.57,54.8709,3,train
1,39.65,481.20,425.65,453.42,55.55,15.37,1.17,17.9829,0,train
2,53.57,886.84,861.37,874.10,25.47,15.24,2.91,44.3484,0,train
3,38.30,317.65,206.12,261.88,111.53,15.61,0.00,0.0000,1,train
4,77.48,786.46,712.02,749.24,74.44,13.03,2.15,28.0145,3,train


## **Splitting the dataset**

In [32]:
train_df = df[df['split'] == 'train'].copy()
val_df   = df[df['split'] == 'validation'].copy()
test_df  = df[df['split'] == 'test'].copy()

# **Scale the dataset**

In [33]:
feature_cols = [
    'temperature_c', 'ldr_left', 'ldr_right', 'avg_light',
    'light_diff', 'voltage_v', 'current_a', 'power_w',
]

In [34]:
scaler = StandardScaler()
scaler.fit(train_df[feature_cols])
train_df[feature_cols] = scaler.transform(train_df[feature_cols])
val_df[feature_cols]   = scaler.transform(val_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

In [35]:
X_train = train_df[feature_cols].values
y_train = train_df['fault_type'].values
X_val   = val_df[feature_cols].values
y_val   = val_df['fault_type'].values
X_test  = test_df[feature_cols].values
y_test  = test_df['fault_type'].values

## **Model training**

In [39]:
model = DecisionTreeClassifier(max_depth=8, random_state=123)
model.fit(X_train, y_train)
print(f"Validation accuracy: {model.score(X_val, y_val)*100:.4f}")

Validation accuracy: 96.1067


In [41]:
print(f"Test accuracy: {model.score(X_test, y_test)*100:.4f}")

Test accuracy: 95.8267


In [ ]:
from sklearn.metrics import classification_report
y_val_pred = model.predict(X_val)
print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      1500
           1       1.00      1.00      1.00      1500
           2       0.99      1.00      0.99      1500
           3       0.92      0.95      0.93      1500
           4       0.98      0.95      0.97      1500

    accuracy                           0.96      7500
   macro avg       0.96      0.96      0.96      7500
weighted avg       0.96      0.96      0.96      7500



In [42]:
print(f"Train accuracy: {model.score(X_train, y_train)*100:.4f}")

Train accuracy: 96.5029


## **Save model**

In [ ]:
with open('../model/solar_fault_model.pkl', 'wb') as f:
    pickle.dump(model, f)

In [ ]:
with open('../model/solar_fault_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)